# Notebook for Milestone - 1

In [2]:
import os
import sys
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
from src.preprocess import clean_text, tokenize, remove_stopwords, get_combined_texts
from src.utils import map_at_3, load_config, set_seed

warnings.filterwarnings('ignore')
load_dotenv('../.env')
set_seed(42)

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
TRAIN_PATH  = '../data/raw/train.csv'



In [3]:
df = pd.read_csv(TRAIN_PATH)
print(f'Shape: {df.shape}')
df.head(3)

Shape: (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


## Q1

In [4]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Answer Distribution ===')
print(df['answer'].value_counts())

=== Dataset Info ===
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      2000 non-null   int64
 1   prompt  2000 non-null   str  
 2   A       2000 non-null   str  
 3   B       2000 non-null   str  
 4   C       2000 non-null   str  
 5   D       2000 non-null   str  
 6   E       2000 non-null   str  
 7   answer  2000 non-null   str  
dtypes: int64(1), str(7)
memory usage: 1.9 MB

=== Missing Values ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== Answer Distribution ===
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [5]:
freq = df['answer'].value_counts()
most_frequent  = freq.index[0]
least_frequent = freq.index[-1]

q1_answer = freq[most_frequent] + freq[least_frequent]

print(f'Answer frequencies:\n{freq}')
print(f'\nMost frequent  : {most_frequent} ({freq[most_frequent]})')
print(f'Least frequent : {least_frequent} ({freq[least_frequent]})')
print(f'\nA1: {q1_answer}')

Answer frequencies:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most frequent  : B (490)
Least frequent : E (324)

A1: 814


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Answer distribution
freq.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Correct Answer Distribution')
axes[0].set_xlabel('Answer')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Prompt length distribution
df['prompt_len'] = df['prompt'].apply(lambda x: len(str(x).split()))
axes[1].hist(df['prompt_len'], bins=30, color='coral', edgecolor='black')
axes[1].set_title('Prompt Length Distribution (words)')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()
print(f'Prompt length stats:\n{df["prompt_len"].describe()}')

# Q2

In [7]:
# Q2: Vocabulary size of cleaned prompts
df['prompt_clean'] = df['prompt'].apply(clean_text)

all_words = []
for text in df['prompt_clean']:
    all_words.extend(text.split())

vocab = set(all_words)
q2_answer = len(vocab)

print(f'Total tokens across all prompts : {len(all_words)}')
print(f'Unique words in cleaned prompts : {q2_answer}')

Total tokens across all prompts : 36293
Unique words in cleaned prompts : 859


# Q3

In [8]:
#Q3: Words in Row ID 1 after stopword removal
row1 = df[df['id'] == 1].iloc[0]
cleaned_row1 = clean_text(row1['prompt'])
tokens_row1  = cleaned_row1.split()
filtered_row1 = [t for t in tokens_row1 if t not in ENGLISH_STOP_WORDS]

print(f'Original prompt  : {row1["prompt"]}')
print(f'Cleaned          : {cleaned_row1}')
print(f'Tokens (raw)     : {tokens_row1}')
print(f'After stop removal: {filtered_row1}')
print(f'Result: {len(filtered_row1)}')

Original prompt  : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Cleaned          : pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options
Tokens (raw)     : ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
After stop removal: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Result: 13


# Q4

In [9]:
# Q4: Vocabulary size of combined texts (prompts + options)
combined_texts = get_combined_texts(df)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)

q4_answer = len(vectorizer.vocabulary_)
print(f'Combined texts fed to vectorizer : {len(combined_texts)}')
print(f'Vocabulary size : {q4_answer}')

Combined texts fed to vectorizer : 12000
Vocabulary size : 2762


# Q5

In [10]:
# Q5: Cosine similarity — prompt vs option A, Row ID 1
row1_prompt   = str(row1['prompt'])
row1_option_a = str(row1['A'])

vec_prompt   = vectorizer.transform([row1_prompt])
vec_option_a = vectorizer.transform([row1_option_a])

q5_answer = round(float(cosine_similarity(vec_prompt, vec_option_a)[0][0]), 4)

print(f'Prompt   : {row1_prompt[:80]}...')
print(f'Option A : {row1_option_a[:80]}...')
print(f'Cosine Similarity: {q5_answer}')

Prompt   : Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
Option A : Martin Heidegger believes that humans exist within a time continuum that is infi...
Cosine Similarity: 0.2328


# Q6

In [11]:
# Q6: % rows where highest cosine sim matches correct answer
correct = 0
total   = len(df)

for _, row in df.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    sims = {}
    for col in OPTION_COLS:
        opt_vec  = vectorizer.transform([str(row[col])])
        sims[col] = float(cosine_similarity(prompt_vec, opt_vec)[0][0])
    best_option = max(sims, key=sims.get)
    if best_option == row['answer']:
        correct += 1

q6_answer = round((correct / total) * 100, 2)
print(f'Correct predictions : {correct} / {total}')
print(f'Result: {q6_answer}%')

Correct predictions : 274 / 2000
Result: 13.7%


# Q7 and 8

In [13]:
# Q7: ground truth = C, prediction = C A B
q7_answer = map_at_3([['C', 'A', 'B']], ['C'])
print(f'Q7 ANSWER: {q7_answer}')

# Q8: ground truth = B, prediction = D B E
q8_answer = map_at_3([['D', 'B', 'E']], ['B'])
print(f'Q8 ANSWER: {q8_answer}')

Q7 ANSWER: 1.0
Q8 ANSWER: 0.5


# Q9

In [15]:
# Q9: Majority class baseline MAP@3
top3_answers = freq.index[:3].tolist()
print(f'Top 3 most frequent answers: {top3_answers}')

majority_preds = [top3_answers for _ in range(len(df))]
ground_truth   = df['answer'].tolist()

q9_answer = round(map_at_3(majority_preds, ground_truth), 4)
print(f'Majority MAP@3: {q9_answer}')

Top 3 most frequent answers: ['B', 'C', 'A']
Majority MAP@3: 0.4213


# Q10

In [16]:
top3_answers = freq.index[:3].tolist()
print(f'Top 3 most frequent answers: {top3_answers}')

majority_preds = [top3_answers for _ in range(len(df))]
ground_truth   = df['answer'].tolist()

q9_answer = round(map_at_3(majority_preds, ground_truth), 4)
print(f'(majority baseline MAP@3): {q9_answer}')

Top 3 most frequent answers: ['B', 'C', 'A']
(majority baseline MAP@3): 0.4213


In [18]:
tfidf_preds = []

for _, row in df.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    sims = {}
    for col in OPTION_COLS:
        opt_vec  = vectorizer.transform([str(row[col])])
        sims[col] = float(cosine_similarity(prompt_vec, opt_vec)[0][0])
    ranked = sorted(sims, key=sims.get, reverse=True)
    tfidf_preds.append(ranked[:3])

q10_answer = round(map_at_3(tfidf_preds, ground_truth), 4)
print(f'(TF-IDF pipeline MAP@3): {q10_answer}')

(TF-IDF pipeline MAP@3): 0.3119
